# HSTU · ARGUS  —  GPU Human Simulator

GPU-optimised implementation of two sequential recommendation architectures
trained as a proxy for a language learner.

**Core abstraction.** Each event is `(context, word, action)` where:
- `context` = surrounding sentence tokens (+ exercise class / CEFR level for synthetic data)
- `word` = the token being tested
- `action` = bool — did the user answer correctly?

**Two GPU architectures (swap with one line):**

| Arch | Attention | FFN | Extras | Objective |
|---|---|---|---|---|
| `hstu_gpu` | SiLU-gated / RoPE | SwiGLU + sparse MoE | Stochastic depth, grad checkpoint | BCE on recall |
| `argus_gpu` | Flash GQA / RoPE | SwiGLU + sparse MoE | Hard-neg NIP, temp annealing | FP + NIP dual-head |

**ARGUS** adds a contrastive Next Item Prediction head (in-batch softmax with
hard-negative re-weighting) alongside the Feedback Prediction head.
At inference only the FP head is used.

**Training features:** BF16 AMP · gradient accumulation · warmup + cosine LR · fused AdamW · OOM guard

**Public API:**
```python
model = build_model_gpu(ModelConfigGPU(arch='argus_gpu'), vocab)

save_model_gpu(model, 'model.pt')
model = load_model_gpu('model.pt')

predictor = HSTUPredictor(model)
p   = predictor.predict_action(history, context, word)
ps  = predictor.predict_action_batch(history, [(word, context), ...])
```

**Layout:** 1 Setup · 2 Data loaders · 3 Vocab + sequences · 4 Models · 5 Training · 6 Eval · 7 API

## 1. Setup and config

In [5]:
# !pip install torch pandas numpy scikit-learn tqdm --quiet
import os, math, json, gzip, urllib.request
from collections import Counter
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss
from tqdm.auto import tqdm

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('./data'); DATA_DIR.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__, '| Device:', DEVICE)

PyTorch: 2.12.0+cpu | Device: cpu


In [6]:
import sys; sys.path.insert(0, str(Path('.').resolve()))

from src.models_gpu import (
    ModelConfigGPU, TrainConfigGPU,
    build_model_gpu, train_gpu,
    save_model_gpu, load_model_gpu,
)
from src.models import (
    HSTUPredictor, HistoryEvent,
    evaluate,
)
from src.slam_loader import load_duolingo_slam
from src.synthetic_dataset import generate_synthetic_events
from src.sequence_dataset import build_dataset, SequenceWindowDataset

from dataclasses import dataclass
from typing import Optional


@dataclass
class DataConfig:
    min_events_per_user:  int = 30
    max_events_per_user:  int = 2000
    min_token_frequency:  int = 5
    val_fraction:         float = 0.1
    subsample_users:      Optional[int] = 5000


# ── Change arch here: 'hstu_gpu' | 'argus_gpu' ──
MODEL_CFG  = ModelConfigGPU(arch='hstu_gpu')
TRAIN_CFG  = TrainConfigGPU(epochs=5)
DATA_CFG   = DataConfig()
print(f'Architecture : {MODEL_CFG.arch.upper()}')
print(f'Device       : {DEVICE}')
print(f'AMP (BF16)   : {TRAIN_CFG.use_amp and DEVICE.type == "cuda"}')

Architecture : HSTU_GPU
Device       : cpu
AMP (BF16)   : False


## 2. Data

**Canonical schema:** `user_id | word | context: list[str] | action: bool | timestamp: int`

Two sources are merged:
- **SLAM** — real Duolingo sentence-level data; context = sentence tokens, exercise = PICK_DEFINITION, no CEFR level.
- **Synthetic** — FSRS-simulated learners; context = `[exercise_class, cefr_level]` for 50 % of users, `[]` for the rest.

See `src/slam_loader.py`, `src/synthetic_dataset.py`, `src/sequence_dataset.py`.

In [7]:
# Load SLAM data  (place en_es.slam.20190204.train under ./data/)
slam_events = load_duolingo_slam(DATA_DIR / 'en_es.slam.20190204.train', max_exercises=500)

# Generate synthetic sessions (equal distribution A1–C2, log-normal session counts)
synthetic_events = generate_synthetic_events(n_humans=1000, n_sessions=500, seed=42)

# Merge, build vocab, create windowed train / val datasets
train_ds, val_ds, vocab = build_dataset(slam_events, synthetic_events, DATA_CFG, MODEL_CFG)

print(f'SLAM events:      {len(slam_events):,}   users: {slam_events.user_id.nunique():,}')
print(f'Synthetic events: {len(synthetic_events):,}   users: {synthetic_events.user_id.nunique():,}')
print(f'Train examples:   {len(train_ds):,}')
print(f'Val examples:     {len(val_ds):,}')
print(f'Vocab size:       {vocab["n_tokens"]}')

SLAM events:      1,697   users: 2
Synthetic events: 117,400   users: 1,000
Train examples:   40,040
Val examples:     11,700
Vocab size:       6428


## 3. Models

All three architectures (`hstu`, `sasrec`, `argus`) are defined in `src/models.py`.
Switch architecture by changing `ModelConfig(arch=...)` in the config cell.

See `src/models.py` for full implementation details.

In [ ]:
model = build_model_gpu(MODEL_CFG, vocab).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'[{MODEL_CFG.arch.upper()}] {n_params/1e6:.3f}M parameters  |  device: {DEVICE}')

## 4. Training loop

**ARGUS** uses a dual-head loss: `FP_loss + nip_weight * NIP_loss`.
The NIP loss is an in-batch sampled softmax over item embeddings.
All other architectures use plain BCE on the recall target.

Training utilities (`argus_loss`, `compute_loss`, `evaluate`, `train`) are in `src/models.py`.

In [ ]:
history = train_gpu(model, train_ds, val_ds, TRAIN_CFG, device=str(DEVICE))
results_df = pd.DataFrame(history)
results_df

## 5. Evaluation: compare architectures

Run the cell below to train **all three architectures** on the same data split and print a comparison table. Each model is initialized freshly so seeds are comparable.

In [ ]:
from dataclasses import asdict

comparison = []
for arch in ['hstu_gpu', 'argus_gpu']:
    torch.manual_seed(0); np.random.seed(0)
    cfg = ModelConfigGPU(**{**asdict(MODEL_CFG), 'arch': arch})
    m   = build_model_gpu(cfg, vocab).to(DEVICE)
    print(f'\n=== {arch.upper()} ({sum(p.numel() for p in m.parameters())/1e6:.2f}M params) ===')
    h = train_gpu(m, train_ds, val_ds, TRAIN_CFG, device=str(DEVICE))
    best = max(h, key=lambda r: r.get('auc', 0))
    best['arch'] = arch
    comparison.append(best)

cmp_df = pd.DataFrame(comparison).set_index('arch')[['auc','log_loss','acc','base_rate','train_loss']]
print('\n── Comparison ─────────────────────────────────')
print(cmp_df.to_string())

In [ ]:
# Forgetting-curve sanity check: mean predicted P(correct) per delta bucket.
# A well-trained model should show higher recall for small deltas (recent events).
def forgetting_curve(m, ds, n_batches=50):
    from torch.utils.data import DataLoader
    m.eval()
    loader = DataLoader(ds, batch_size=64, shuffle=True)
    by_bucket = {}
    with torch.no_grad():
        for i, b in enumerate(loader):
            if i >= n_batches: break
            b = {k: v.to(DEVICE) for k, v in b.items()}
            probs = torch.sigmoid(m(b)).cpu().numpy()
            for d, p, y in zip(b['target_delta'].cpu().numpy(), probs,
                                b['target_label'].cpu().numpy()):
                by_bucket.setdefault(int(d), {'pred': [], 'label': []})
                by_bucket[int(d)]['pred'].append(float(p))
                by_bucket[int(d)]['label'].append(float(y))
    import numpy as np
    return pd.DataFrame([{'delta_bucket': d, 'n': len(v['pred']),
                           'mean_pred': np.mean(v['pred']), 'mean_label': np.mean(v['label'])}
                          for d, v in sorted(by_bucket.items())])

forgetting_curve(model, val_ds)

## 6. Save / load + `predict_action` API

`hstu.save` / `hstu.load` work identically for all three architectures.
The saved file includes the `arch` field so the right class is instantiated automatically.

`HSTUPredictor` and `HistoryEvent` are imported from `src/models.py`.

In [ ]:
SAVE_PATH = './model.pt'
save_model_gpu(model, SAVE_PATH)
print(f'Saved [{MODEL_CFG.arch.upper()}] to {SAVE_PATH} '
      f'({Path(SAVE_PATH).stat().st_size/1e3:.0f} KB)')

reloaded  = load_model_gpu(SAVE_PATH, map_location=str(DEVICE))
predictor = HSTUPredictor(reloaded)

history_events = [
    HistoryEvent('cat',   ['the', 'cat', 'sleeps'],       True,  0),
    HistoryEvent('runs',  ['the', 'dog', 'runs'],          False, 3600),
    HistoryEvent('apple', ['she', 'eats', 'an', 'apple'], True,  7200),
]

p = predictor.predict_action(history_events, ['he', 'reads', 'a', 'book'],
                              'book', now_timestamp=10800)
print(f"\nP(correct on 'book') = {p:.4f}")

candidates = [
    ('book',  ['he', 'reads', 'a', 'book']),
    ('cat',   ['the', 'cat', 'sleeps']),
    ('apple', ['she', 'eats', 'an', 'apple']),
    ('blorg', ['an', 'entirely', 'new', 'sentence']),
]
probs = predictor.predict_action_batch(history_events, candidates, now_timestamp=10800)
print('\nBatch predictions:')
for (w, _), prob in zip(candidates, probs):
    print(f'  {w:10s}  P(correct) = {prob:.4f}')

## Plugging in a new dataset

```python
def load_my_app(path):
    df = pd.read_json(path, lines=True)
    return pd.DataFrame({
        'user_id':   df['user'],
        'word':      df['target_word'],
        'context':   df['sentence_tokens'],   # list[str]; [] is fine
        'action':    df['was_correct'].astype(bool),
        'timestamp': df['ts'].astype('int64'),
    })

my_events = load_my_app('events.jsonl')
train_ds, val_ds, vocab = build_dataset(my_events, None, DATA_CFG, MODEL_CFG)
model = build_model(MODEL_CFG, vocab)
```

## Knobs cheatsheet

| Goal | Change |
|---|---|
| Switch architecture | `ModelConfig(arch='sasrec')` |
| Fatter model | `d_model=128, n_heads=4, n_layers=4` |
| More context | `max_context_tokens=32` |
| Longer histories | `max_seq_len=256` |
| ARGUS NIP balance | `argus_nip_weight=0.3` |
| Softer predictions | `predictor.predict_action(..., temperature=2.0)` |
| Faster experiments | `DataConfig(subsample_users=500)` |
| More synthetic data | `generate_synthetic_events(n_humans=1000, n_sessions=5000)` |